<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/02_time_series/03_classification_quest_gunpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔫 Quest · Time-series classification: Gun or Point?

**University of California, Riverside, 2003.** In a lab, two actors stand in front of a camera. Again and again they
perform one of two gestures: draw a replica gun from a hip holster, aim it at a target and put it back
(**Gun-Draw**), or simply raise the hand and point a finger at the target (**Point**). A tracker records the
horizontal position of the hand at each video frame: every gesture becomes a curve of 150 numbers.

Can a machine tell the two gestures apart **only from the shape of the curve**? This tiny dataset, *GunPoint*, became
one of the stars of the **UCR Time Series Archive**, created in 2002 by Eamonn Keogh's group to let researchers
compare time-series algorithms on the same data. With 128 datasets since 2018 (ECGs, insect wing beats, power
consumption, spectrographs of coffee beans…), it is the "ImageNet" of time-series classification.

**Your mission:** classify each curve as Gun-Draw or Point, a **time-series classification** task.
**Metric:** accuracy on the official test split (so your score is comparable with research papers).

| You will practise | Concept |
|---|---|
| The shape of time-series data | `(n_cases, n_channels, n_timepoints)` |
| Nearest neighbours | Euclidean vs **Dynamic Time Warping** distance |
| Feature-based classification | turning a curve into a handful of numbers |
| Convolution-based classification | **MiniRocket**, the fast state of the art |
| Comparing accuracy *and* cost | the right tool for the right budget |

🧩 Complete the `# TODO` cells, run the `check(...)` cells, reveal a solution if you are stuck.

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aeon"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeon.datasets import load_classification

quest("Gun or Point?", chapter="Track 2 · Time series · Classification", xp=300)

## 1. Load the archive 📥

`aeon` (a scikit-learn-like library dedicated to time series) downloads UCR datasets with their **official**
train/test split.

In [ ]:
X_train, y_train = load_classification("GunPoint", split="train", extract_path=str(DATA_DIR / "ucr"))
X_test, y_test = load_classification("GunPoint", split="test", extract_path=str(DATA_DIR / "ucr"))
labels = {"1": "Gun-Draw", "2": "Point"}
print(type(X_train), X_train.shape, X_test.shape)
print("train counts:", pd.Series(y_train).map(labels).value_counts().to_dict())

Time-series collections are 3-D arrays: `(n_cases, n_channels, n_timepoints)`. Here each case has **1 channel**
(the hand's x-position). A smartphone accelerometer would have 3 channels (x, y, z): we will see that in the next
application.

### 🎯 Exercise 1: read the shape
How many training curves, and how many time points per curve?

In [ ]:
n_train_cases = ...   # TODO
series_length = ...   # TODO

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
n_train_cases = X_train.shape[0]
series_length = X_train.shape[2]
```

</details>

In [ ]:
check(n_train_cases == 50 and series_length == 150,
      "50 training curves of 150 points (and 150 test curves). Tiny training set: a perfect playground for careful methods.",
      "Use X_train.shape: (n_cases, n_channels, n_timepoints).")

## 2. Look at the curves 👀

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, cls in zip(axes, ["1", "2"]):
    for curve in X_train[y_train == cls][:12, 0]:
        ax.plot(curve, alpha=0.5)
    ax.set(title=f"Class {cls}: {labels[cls]}", xlabel="time (video frame)")
axes[0].set_ylabel("hand x-position (normalised)")
plt.tight_layout(); plt.show()

### 🎯 Exercise 2: the average gesture
Compute the mean curve of each class (an array of 150 values each) and plot both. Where do they differ?

In [ ]:
mean_gun = ...     # TODO: hint: X_train[y_train == "1", 0].mean(axis=0)
mean_point = ...   # TODO
plt.plot(mean_gun, label="Gun-Draw (mean)"); plt.plot(mean_point, label="Point (mean)")
plt.legend(); plt.title("Average gesture per class"); plt.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
mean_gun = X_train[y_train == "1", 0].mean(axis=0)
mean_point = X_train[y_train == "2", 0].mean(axis=0)
plt.plot(mean_gun, label="Gun-Draw (mean)"); plt.plot(mean_point, label="Point (mean)")
plt.legend(); plt.title("Average gesture per class"); plt.show()
```

</details>

In [ ]:
check(np.shape(mean_gun) == (150,) and np.shape(mean_point) == (150,),
      "Look at the beginning and the end of the 'Gun' curve: a small dip when the hand reaches the holster and when it puts the gun back. That subtle shape is the whole signal!",
      "Each mean curve should be a 1-D array of 150 values.")

## 3. Nearest neighbour: "you are like the curve you resemble most" 📏

The simplest classifier: for a new curve, find the **most similar** training curve and copy its label (1-NN).
Everything depends on how we measure *similarity*.

**Euclidean distance** compares the curves point by point: frame 40 with frame 40. It treats the curve as a plain
vector of 150 features, so any scikit-learn classifier can use it after flattening the array to 2-D.

### 🎯 Exercise 3: 1-NN with Euclidean distance

In [ ]:
lb = Leaderboard("gunpoint", metric="accuracy")
def timed(name, model, Xtr, Xte):
    t0 = time.perf_counter()
    model.fit(Xtr, y_train)
    acc = (model.predict(Xte) == y_test).mean()
    lb.add(name, acc, seconds=round(time.perf_counter() - t0, 2))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
X_train_2d = ...   # TODO: flatten to (n_cases, n_timepoints). hint: X_train[:, 0, :] or X_train.reshape(len(X_train), -1)
X_test_2d = ...    # TODO
timed("1-NN Euclidean", KNeighborsClassifier(n_neighbors=1), X_train_2d, X_test_2d)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.neighbors import KNeighborsClassifier
X_train_2d = X_train.reshape(len(X_train), -1)
X_test_2d = X_test.reshape(len(X_test), -1)
timed("1-NN Euclidean", KNeighborsClassifier(n_neighbors=1), X_train_2d, X_test_2d)
```

</details>

In [ ]:
check(X_train_2d.shape == (50, 150) and lb.best()["accuracy"] > 0.9,
      "~91% with the dumbest possible model. In time-series research, 1-NN is THE baseline every new method must beat.",
      "Flatten to 2-D arrays of shape (50, 150) and (150, 150).")

## 4. Dynamic Time Warping: let the curves stretch ⏱️

Two people drawing a gun don't move at exactly the same speed: one gesture may start 5 frames later or last longer.
Euclidean distance punishes that harshly. **Dynamic Time Warping (DTW)**, introduced for **speech recognition** in
the 1970s (people pronounce words at different speeds!), finds the best *alignment* between two curves, allowing
time to stretch and compress, before measuring the distance.

In [ ]:
from aeon.distances import dtw_distance, dtw_alignment_path, euclidean_distance
a, b = X_train[0, 0], X_train[1, 0]
path, _ = dtw_alignment_path(a, b)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(a, label="curve A"); ax.plot(b + 1.5, label="curve B (shifted up for display)")
for i, j in path[::4]:
    ax.plot([i, j], [a[i], b[j] + 1.5], color="gray", lw=0.5)
ax.set(title=f"DTW alignment (grey lines)  ·  Euclidean = {euclidean_distance(a, b):.2f}, DTW = {dtw_distance(a, b):.2f}")
ax.legend(); plt.show()

### 🎯 Exercise 4: 1-NN with DTW
`aeon` classifiers take the 3-D array directly. Use `KNeighborsTimeSeriesClassifier(distance="dtw", n_neighbors=1)`.

In [ ]:
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
knn_dtw = ...   # TODO
timed("1-NN DTW", knn_dtw, X_train, X_test)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
knn_dtw = KNeighborsTimeSeriesClassifier(distance="dtw", n_neighbors=1)
timed("1-NN DTW", knn_dtw, X_train, X_test)
```

</details>

In [ ]:
lb.show()

Surprise: on GunPoint, unconstrained DTW is **not** better than Euclidean, and it is much slower. The curves are
already well aligned (the recording always starts at the same moment), and too much warping lets a "Point" curve
stretch itself into looking like a "Gun" curve. DTW usually shines when timing varies; researchers often limit the
warping to a window (`distance_params={"window": 0.1}`). **No method wins everywhere: always test.**

In [ ]:
checkpoint("gunpoint/distances", xp=60)

## 5. Feature-based: summarise each curve in a few numbers 🧮

Another strategy: describe each curve with **features** (statistics), then use any tabular classifier, exactly like
in the tabular track. It is fast and often interpretable.

### 🎯 Exercise 5: hand-crafted features + Random Forest
Write `simple_features` returning, for each curve: mean, std, min, max, the index of the max (`argmax`), and the mean of
the absolute differences between consecutive points (how "agitated" the curve is).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def simple_features(X3d: np.ndarray) -> np.ndarray:
    x = X3d[:, 0, :]
    feats = [
        x.mean(axis=1),
        ...,   # TODO: std
        ...,   # TODO: min
        ...,   # TODO: max
        ...,   # TODO: argmax  (hint: x.argmax(axis=1))
        ...,   # TODO: mean absolute difference (hint: np.abs(np.diff(x, axis=1)).mean(axis=1))
    ]
    return np.column_stack(feats)

F_train, F_test = simple_features(X_train), simple_features(X_test)
timed("Simple stats + RandomForest", RandomForestClassifier(n_estimators=500, random_state=SEED), F_train, F_test)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from sklearn.ensemble import RandomForestClassifier

def simple_features(X3d: np.ndarray) -> np.ndarray:
    x = X3d[:, 0, :]
    feats = [
        x.mean(axis=1),
        x.std(axis=1),
        x.min(axis=1),
        x.max(axis=1),
        x.argmax(axis=1),
        np.abs(np.diff(x, axis=1)).mean(axis=1),
    ]
    return np.column_stack(feats)

F_train, F_test = simple_features(X_train), simple_features(X_test)
timed("Simple stats + RandomForest", RandomForestClassifier(n_estimators=500, random_state=SEED), F_train, F_test)
```

</details>

In [ ]:
check(F_train.shape == (50, 6), "6 numbers per curve instead of 150. The score drops: our intuitive statistics miss the subtle dips near the holster. Smarter features next!",
      "simple_features should return an array of shape (n_cases, 6).")

Hand-crafted features depend on your intuition. **catch22** (Lubba et al., 2019) is a set of 22 features selected
from more than 7 000 candidates as the most useful across the UCR archive: autocorrelations, entropy, outlier
timing… Let `aeon` compute them.

In [ ]:
from aeon.transformations.collection.feature_based import Catch22
c22 = Catch22()
C_train, C_test = c22.fit_transform(X_train), c22.transform(X_test)
timed("catch22 + RandomForest", RandomForestClassifier(n_estimators=500, random_state=SEED), C_train, C_test)

## 6. MiniRocket: thousands of random convolutions 🚀

**ROCKET** (Dempster et al., 2020) had a crazy idea: generate **10 000 random convolution kernels** (small pattern
detectors, like in image CNNs), slide them along the series, keep two numbers per kernel (the max and the proportion
of positive values), and feed everything to a simple linear classifier. No training of the kernels at all!
It matched the best (and very slow) methods of the time while being orders of magnitude faster.
**MiniRocket** (2021) made it almost deterministic and even faster. It is the go-to first model for time-series
classification today.

### 🎯 Exercise 6: MiniRocket

In [ ]:
from aeon.classification.convolution_based import MiniRocketClassifier
minirocket = ...   # TODO: MiniRocketClassifier(random_state=SEED)
timed("MiniRocket", minirocket, X_train, X_test)
lb.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
from aeon.classification.convolution_based import MiniRocketClassifier
minirocket = MiniRocketClassifier(random_state=SEED)
timed("MiniRocket", minirocket, X_train, X_test)
lb.show()
```

</details>

In [ ]:
check(lb.best()["model"] == "MiniRocket" and lb.best()["accuracy"] > 0.97,
      "MiniRocket tops the board: random convolutions + a linear model nail this problem.",
      "MiniRocket should reach ~99% accuracy. Did you fit it on the 3-D X_train?")
checkpoint("gunpoint/minirocket", xp=100)

## 7. Which curves fool the models? 🔎

In [ ]:
pred_mr = minirocket.predict(X_test)
pred_ed = KNeighborsClassifier(n_neighbors=1).fit(X_train_2d, y_train).predict(X_test_2d)
wrong = np.where(pred_ed != y_test)[0][:6]
fig, axes = plt.subplots(1, len(wrong), figsize=(3 * len(wrong), 2.6), sharey=True)
for ax, i in zip(np.atleast_1d(axes), wrong):
    ax.plot(X_test[i, 0], color="tab:red")
    ax.set_title(f"true {labels[y_test[i]]}\n1-NN: {labels[pred_ed[i]]} · MR: {labels[pred_mr[i]]}", fontsize=8)
plt.suptitle("Test curves misclassified by 1-NN Euclidean", y=1.05); plt.show()

## 🏁 Quest complete!

You met the three big families of time-series classifiers:
1. **Distance-based** (1-NN Euclidean / DTW): simple, strong baselines; DTW handles timing differences.
2. **Feature-based** (hand-made stats, catch22): fast and interpretable.
3. **Convolution-based** (ROCKET / MiniRocket): the fast state of the art, and the idea behind deep 1D-CNNs.

### 🚀 Your turn (bonus XP)
1. Try DTW with a warping window: `KNeighborsTimeSeriesClassifier(distance="dtw", distance_params={"window": 0.1})`.
2. Replay the whole notebook on another UCR dataset: `"ECG200"` (heartbeats), `"ItalyPowerDemand"` (electricity),
   `"Coffee"` (spectrographs of Arabica vs Robusta beans). Does the ranking change?
3. Add the 22 catch22 features to your 6 simple features. Better together?
4. Next: `04_classification_app_human_activity.ipynb`: your smartphone knows if you are walking, sitting or climbing
   stairs.

In [ ]:
checkpoint("gunpoint/complete", xp=140)
badge("Shape Whisperer", "〰️")
progress()